# Phase 5 · Legal Statute Identification — Architecture-3 E-LegalFormer (frozen) + AdaLoRA + layer mix + label attention

| | |
|---|---|
| **Codabench** | Macro-F1 **0.2817** · Micro-F1 0.368 · Subset acc. 0.0113 (rank 5 of 7 in Phase 5) |
| **Own dev / validation** | dev macro-F1 0.3105, micro-F1 0.3905, subset acc. 0.0100 (best epoch 4) |
| **Backbone** | Architecture-3 E-LegalFormer, GQA (125M) |

**The task, in simple words:** give every case **all** statutes (out of 100) that apply to it — a multi-label problem.

**How this notebook solves it:**
1. Cut each document into up to 8 chunks of 512 tokens (start, middle and end of long documents).
2. The Phase-1 model gives a vector for every token (a learned mix of all its layers).
3. Two classifier parts add up to 100 statute scores: one from the average token vector, and *label attention*, where every statute looks for the words that signal it.
4. One threshold per statute (chosen on dev) turns the 100 probabilities into the predicted set.

**What is special in this run:** **AdaLoRA** — like LoRA, but the rank budget moves during training to the layers that need it most (unimportant directions are switched off).

*Exact setting:* AdaLoRA: rank 12 → average 8, importance-based pruning of singular triplets.

**To run it:** Attach the tokenizer notebook's output and the Phase-1 checkpoint as Kaggle inputs, add the Hugging Face token as the Kaggle secret **`HF`**, pick a GPU (T4), then *Run All*. Paths are in the configuration cell; if a checkpoint is not found there, the notebook searches `/kaggle/input`. Test labels are never used.

Every step below starts with a short explanation: first in simple words, then how it works exactly.

### CELL 1 · Imports + reproducibility + device

**In simple words:** Load the Python libraries, fix the random seed (so a re-run gives the same result) and pick the GPU.

**How it works:** Seeds for `random`, NumPy and PyTorch; `DEVICE` is the GPU when one is attached.

In [1]:
# ============================================================
# CELL 1 — IMPORTS + REPRODUCIBILITY + DEVICE
# ============================================================

import os
import gc
import sys
import json
import math
import time
import random
import zipfile
import importlib.util

from pathlib import Path
from collections import Counter

import numpy as np

# less GPU memory fragmentation (must be set before importing torch)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import f1_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("Device :", DEVICE)

if DEVICE.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Device : cuda
GPU    : Tesla T4


### CELL 2 · Configuration

**In simple words:** Every setting of this run in one place: input paths, sizes, learning rates, epochs. To change the experiment, change only this cell.

**How it works:** The comment block at the top summarises the whole method. Mixed precision: float16 + GradScaler on T4 / P100, bfloat16 on A100. `DEBUG_MODE = True` (where present) runs a few-minute check on a small subset.

In [2]:
# ============================================================
# CELL 2 — CONFIGURATION
# Phase 5 (Legal Statute Identification, MULTI-LABEL): Architecture-3 E-LegalFormer + AdaLoRA
#
#   document -> placeholders shortened (<ENTITY> -> X, <SECTION> -> section, <ACT> -> Act)
#            -> up to 8 chunks of 512 tokens (the whole document if it fits in 4096
#               tokens, else 8 chunks spread over its start, middle and end)
#            -> every chunk through the Phase-1 model -> token vector = learned MIX of ALL layers
#   classifier (100 statutes) = two parts that are added:
#     LINEAR path : mean of the token vectors of the document -> 100 scores
#                   (STARTS as a logistic regression on the frozen features, CELL 11)
#     LABEL ATTENTION: every statute has its own attention over ALL tokens of the document
#                   (it learns which words point to that statute) -> 100 scores (starts at 0)
#   -> 100 probabilities -> one threshold per statute (tuned on dev) -> set of statute ids
#
# Whitespace tokens: the Phase-1 tokenizer makes every space its own token (about half of
# all tokens). They stay in the input (as in pretraining) but are NOT pooled / attended.
#
# Trained on TRAIN only; dev picks the best epoch and the thresholds.
# Reference (real data): word counts (TF-IDF) reach dev macro-F1 0.394;
# Arch-1 + LoRMA v1 (mean pooling only) 0.288.
# ============================================================

RUN_NAME = "p5_arch3_adalora"


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

TOKENIZER_DIR = Path(
    "/kaggle/input/notebooks/ritikguptatab/updated-bpe-tokenizer/"
    "legal_tokenizer_v3/tokenizer/final_tokenizer"
)

# Phase-1 Architecture-3 checkpoint. If wrong, CELL 9 searches /kaggle/input for it.
CHECKPOINT_PATH = Path(
    "/kaggle/input/notebooks/ritikguptatab/proj-cap-p-1-archi-3/model3_checkpoints/best_model.pt"
)

# Phase-5 data: None = download from Hugging Face (Kaggle secret "HF"),
# or the folder that contains Train/ and Test/, e.g. Path("/kaggle/input/phase-5-lsi/Phase_5_LSI")
LOCAL_DATA_DIR = None
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

OUTPUT_DIR = Path(f"/kaggle/working/{RUN_NAME}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Input
# ------------------------------------------------------------

CHUNK_LEN = 512              # tokens per chunk (every chunk goes through the Phase-1 model separately)
MAX_CHUNKS = 8               # up to 8 x 512 = 4096 tokens per document (covers ~75% of documents fully)

# the masked mentions take several tokens and the Phase-1 model never saw them:
# replaced by short, ordinary words (CELL 3 shows how they are tokenized)
PLACEHOLDERS = {"<ENTITY>": "X", "<SECTION>": "section", "<ACT>": "Act"}

POOL_SKIP_SPACES = True      # whitespace tokens are not pooled / attended (they stay in the input)


# ------------------------------------------------------------
# AdaLoRA (Zhang et al., 2023): LoRA in SVD form, with the rank ADAPTED during training
#     W' = W + alpha/r * P diag(E) Q       P: out x r, Q: r x in, E: r "singular values" (start at 0)
#   - Phase-1 weights W stay FROZEN; P, E, Q train
#   - every triplet (P_i, E_i, Q_i) has an importance = |weight x gradient| (smoothed);
#     every ADALORA_DELTA_T updates the least important triplets over ALL layers are switched
#     off, until on average ADALORA_TARGET_RANK of ADALORA_INIT_RANK remain (cubic schedule)
#     -> layers that matter keep more rank
#   - + orthogonality regularization ||P^T P - I||_F + ||Q Q^T - I||_F
# ------------------------------------------------------------

ADALORA_TARGETS = ["attention.q_proj", "attention.k_proj", "attention.v_proj", "attention.o_proj",
                   "ffn.gate_proj", "ffn.up_proj", "ffn.down_proj"]     # every linear layer of every block
ADALORA_INIT_RANK = 12
ADALORA_TARGET_RANK = 8          # average rank at the end
ADALORA_ALPHA = 24
ADALORA_DROPOUT = 0.05
ADALORA_T_INIT = 200             # updates before the first cut
ADALORA_T_FINAL = 500            # last updates: the ranks stay fixed
ADALORA_DELTA_T = 10             # updates between cuts
ADALORA_BETA = 0.85              # smoothing of the importance
ADALORA_ORTH_WEIGHT = 0.5

LR_PEFT = 5e-4


# ------------------------------------------------------------
# Head + loss
# ------------------------------------------------------------

POOL_SKIP_TOKENS = 4         # the first tokens of a chunk are not pooled (they behave abnormally in LMs)
LABEL_ATTENTION_DIM = 256    # size of the attention keys of the label attention
HEAD_DROPOUT = 0.1
LR_HEAD = 1e-3

# BCE with a weight on the positives of every statute: (negatives / positives) ** POWER,
# so rarer statutes count more (macro-F1 counts every statute equally)
POS_WEIGHT_POWER = 0.5
POS_WEIGHT_MAX = 10.0


# ------------------------------------------------------------
# Training (TRAIN data only)
#   epoch 0 = the start (logistic regression on the frozen features)
#   epoch 1 = head only (AdaLoRA matrices wait, the Phase-1 model runs without gradients: fast)
#   epoch 2.. = head + AdaLoRA matrices
# ------------------------------------------------------------

EPOCHS = 4
EPOCH_DOCS = 10000           # documents per epoch (all train documents are used before any repeats)
BATCH_DOCS = 2               # documents per forward pass (each up to 8 chunks). If CUDA out-of-memory: 1
GRAD_ACCUM = 8               # -> 16 documents per weight update
HEAD_ONLY_EPOCHS = 1
PATIENCE = 2                 # stop if dev macro-F1 does not improve for 2 epochs
WARMUP_RATIO = 0.05
GRAD_CLIP = 1.0

EVAL_CHUNKS = 48             # chunks per forward pass when predicting
DEV_EVAL_DOCS = 3000         # dev documents scored after every epoch (the full dev set at the end, CELL 14)

# Before training (CELL 11): token statistics from STATS_DOCS train documents, and a probe of
# every layer + the start of the linear path on PROBE_DOCS train documents.
STATS_DOCS = 800
PROBE_DOCS = 4000


# ------------------------------------------------------------
# Debug mode (quick run on a few documents)
# ------------------------------------------------------------

DEBUG_MODE = False
DEBUG_DOCS = 200             # per split

if DEBUG_MODE:
    EPOCHS = 2
    EPOCH_DOCS = 64
    PROBE_DOCS = 100
    STATS_DOCS = 50
    DEV_EVAL_DOCS = 100


# ------------------------------------------------------------
# Mixed precision (T4 / P100 -> float16 + GradScaler, A100 -> bfloat16)
# ------------------------------------------------------------

USE_AMP = DEVICE.type == "cuda"

if USE_AMP and torch.cuda.get_device_capability()[0] >= 8:
    AMP_DTYPE = torch.bfloat16
else:
    AMP_DTYPE = torch.float16

USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16


def autocast():
    return torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP)


print("Run       :", RUN_NAME)
print("Reads     :", f"up to {MAX_CHUNKS} chunks x {CHUNK_LEN} tokens per document")
print("Output    :", OUTPUT_DIR)
print("AMP dtype :", AMP_DTYPE if USE_AMP else "disabled")


Run       : p5_arch3_adalora
Reads     : up to 8 chunks x 512 tokens per document
Output    : /kaggle/working/p5_arch3_adalora
AMP dtype : torch.float16


### CELL 3 · Load the Phase-1 tokenizer + fast encoding

**In simple words:** Load our own Phase-1 BPE tokenizer — the same one the Phase-1 model was trained with. Any other tokenizer would give the model token ids it has never seen.

**How it works:** `tokenizer.py` from the tokenizer notebook. Long documents: only the needed part is tokenized, and every pre-token piece is BPE-encoded once and cached, which makes tokenizing thousands of judgments fast.

In [3]:
# ============================================================
# CELL 3 — LOAD THE PHASE-1 TOKENIZER + FAST ENCODING
#
# Documents are long, so only the parts we need are tokenized,
# and every pre-token piece is BPE-encoded only once (cache).
# Placeholders (<ENTITY>, <SECTION>, <ACT>) are replaced first.
# ============================================================

spec = importlib.util.spec_from_file_location("legal_tokenizer_module", TOKENIZER_DIR / "tokenizer.py")
tok_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tok_module)

tokenizer, tokenizer_config = tok_module.load_tokenizer(TOKENIZER_DIR)

PAD_ID = tokenizer.pad_id
UNK_ID = tokenizer.unk_id
VOCAB_SIZE = tokenizer.get_vocab_size()

print("Vocabulary size:", VOCAB_SIZE, "| PAD / UNK:", PAD_ID, UNK_ID)


piece_cache = {}      # pre-token piece -> token ids


def encode_fast(text):
    """Same ids as tokenizer.encode(text), but every piece is encoded only once."""

    ids = []

    for piece in tok_module.PRETOKEN_PATTERN.findall(tok_module.normalize_text(text)):

        if piece not in piece_cache:
            piece_cache[piece] = tokenizer.encode(piece)

        ids.extend(piece_cache[piece])

    return ids


def head_tokens(text, n):
    """First n tokens of text (tokenizes only the beginning, cut at a space so no word is split)."""

    chars = n * 8

    while True:

        if chars >= len(text):
            return encode_fast(text)[:n]

        part = text[:chars]
        part = part[:max(part.rfind(" "), 1)]

        ids = encode_fast(part)
        if len(ids) >= n:
            return ids[:n]

        chars *= 2


def tail_tokens(text, n):
    """Last n tokens of text (tokenizes only the end, cut at a space so no word is split)."""

    chars = n * 8

    while True:

        if chars >= len(text):
            return encode_fast(text)[-n:]

        part = text[-chars:]
        part = part[max(part.find(" "), 0):]

        ids = encode_fast(part)
        if len(ids) >= n:
            return ids[-n:]

        chars *= 2


demo = "The appeal is allowed. The order of the High Court dated 12.03.2010 is set aside."
assert encode_fast(demo) == tokenizer.encode(demo)
print("Demo:", [tokenizer.id_to_token[i] for i in encode_fast(demo)][:12])

def clean(text):
    """Placeholders -> short ordinary words (PLACEHOLDERS, CELL 2)."""
    for old, new in PLACEHOLDERS.items():
        text = text.replace(old, new)
    return text


print("\nHow the placeholders are tokenized:")
for old, new in PLACEHOLDERS.items():
    print(f"  {old:10s} -> {len(encode_fast(old))} tokens {[tokenizer.id_to_token[i] for i in encode_fast(old)]} | "
          f"replaced by {new!r} -> {len(encode_fast(' ' + new))} token(s)")


# whitespace-only tokens (" ", "\n", ...): kept in the input, not pooled (POOL_SKIP_SPACES)
def token_text(i):
    try:
        return tokenizer.id_to_token[i]
    except (KeyError, IndexError):
        return None


SPACE_TOKEN_IDS = [i for i in range(VOCAB_SIZE)
                   if i != PAD_ID and isinstance(token_text(i), str) and token_text(i).strip() == ""]

print(f"\nWhitespace-only tokens in the vocabulary: {len(SPACE_TOKEN_IDS)}, e.g. "
      f"{[token_text(i) for i in SPACE_TOKEN_IDS[:6]]}")


Vocabulary size: 32004 | PAD / UNK: 0 1
Demo: ['The', ' ', 'appeal', ' ', 'is', ' ', 'allow', 'ed', '.', ' ', 'The', ' ']

How the placeholders are tokenized:
  <ENTITY>   -> 5 tokens ['<', 'EN', 'TI', 'TY', '>'] | replaced by 'X' -> 2 token(s)
  <SECTION>  -> 3 tokens ['<', 'SECTION', '>'] | replaced by 'section' -> 2 token(s)
  <ACT>      -> 3 tokens ['<', 'ACT', '>'] | replaced by 'Act' -> 2 token(s)

Whitespace-only tokens in the vocabulary: 3, e.g. [' ', '\n', '\n\n']


### CELL 4 · Phase-5 data files (download needs the Kaggle secret "HF")

**In simple words:** Download the official data of this phase from Hugging Face.

**How it works:** The dataset is gated: the token comes from the Kaggle secret `HF` (never written in the notebook). Files are cached, so a re-run does not download again.

In [4]:
# ============================================================
# CELL 4 — PHASE-5 DATA FILES  (download needs the Kaggle secret "HF")
# ============================================================

PHASE5_FILES = {
    "train": "Phase_5_LSI/Train/train.json",
    "dev":   "Phase_5_LSI/Train/dev.json",
    "test":  "Phase_5_LSI/Test/test.json",
}

DATA_FILES = {}

if LOCAL_DATA_DIR is not None:

    for name, filename in PHASE5_FILES.items():
        DATA_FILES[name] = Path(LOCAL_DATA_DIR) / filename.replace("Phase_5_LSI/", "")

else:

    from huggingface_hub import hf_hub_download
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF")

    for name, filename in PHASE5_FILES.items():
        DATA_FILES[name] = hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename,
                                           repo_type="dataset", token=HF_TOKEN)

for name, path in DATA_FILES.items():
    print(f"{name:6s} -> {path}  ({os.path.getsize(path) / 1e6:.0f} MB)")


train  -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_5_LSI/Train/train.json  (299 MB)
dev    -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_5_LSI/Train/dev.json  (79 MB)
test   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_5_LSI/Test/test.json  (98 MB)


### CELL 5 · Load documents + statutes

**In simple words:** Read the documents and their statute ids (several per document).

**How it works:** Every document gets a 0 / 1 vector over the 100 statutes.

In [5]:
# ============================================================
# CELL 5 — LOAD DOCUMENTS + STATUTES
#
# Every document: {"id", "text", "labels" (sorted, unique statute ids; None for test),
#                  "y" (0/1 vector over the statutes)}
# ============================================================

def read_json(path):
    """A JSON file, or a JSON-lines file (one document per line)."""

    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return [json.loads(line) for line in content.splitlines() if line.strip()]


def load_docs(path, with_labels=True):

    docs = []

    for i, item in enumerate(read_json(path)):
        docs.append({
            "id": str(item.get("id", i)),
            "text": str(item.get("text", "") or ""),
            "labels": sorted(set(int(l) for l in item.get("labels", []))) if with_labels else None,
        })

    if DEBUG_MODE:
        docs = docs[:DEBUG_DOCS]

    return docs


train_docs = load_docs(DATA_FILES["train"])
dev_docs = load_docs(DATA_FILES["dev"])
test_docs = load_docs(DATA_FILES["test"], with_labels=False)     # test labels are private: never read

ids = [d["id"] for d in test_docs]
assert len(ids) == len(set(ids)), "test ids are not unique"


# ------------------------------------------------------------
# Statutes: the label columns
# ------------------------------------------------------------

LABELS = sorted({l for d in train_docs + dev_docs for l in d["labels"]})   # statute ids, e.g. 0 .. 99
NUM_LABELS = len(LABELS)
LABEL_INDEX = {l: j for j, l in enumerate(LABELS)}

missing = sorted({l for d in dev_docs for l in d["labels"]} - {l for d in train_docs for l in d["labels"]})
if missing:
    print(f"WARNING: dev statutes never seen in train (only possible in DEBUG_MODE): {missing}")

for d in train_docs + dev_docs:
    d["y"] = np.zeros(NUM_LABELS, dtype=np.float32)
    d["y"][[LABEL_INDEX[l] for l in d["labels"]]] = 1.0

Y_TRAIN = np.stack([d["y"] for d in train_docs])
POSITIVES = Y_TRAIN.sum(axis=0)

# weight on the positives of every statute (BCE): (negatives / positives) ** POWER
POS_WEIGHT = torch.tensor(np.clip(((len(train_docs) - POSITIVES) / np.maximum(POSITIVES, 1)) ** POS_WEIGHT_POWER,
                                  1.0, POS_WEIGHT_MAX), dtype=torch.float32)

print(f"{'split':6s} {'docs':>7s} {'statutes per doc':>17s} {'chars (median / 90%)':>22s}")
for name, docs in [("train", train_docs), ("dev", dev_docs), ("test", test_docs)]:
    chars = np.array([len(d["text"]) for d in docs])
    per_doc = np.mean([len(d["labels"]) for d in docs]) if docs[0]["labels"] is not None else float("nan")
    print(f"{name:6s} {len(docs):7,d} {per_doc:17.2f} {np.median(chars):11,.0f} / {np.percentile(chars, 90):8,.0f}")

print(f"\n{NUM_LABELS} statutes (ids {LABELS[0]} .. {LABELS[-1]}) | train documents per statute: "
      f"min {int(POSITIVES.min())}, median {int(np.median(POSITIVES))}, max {int(POSITIVES.max())}")
print(f"positive weights: min {POS_WEIGHT.min():.2f}, median {POS_WEIGHT.median():.2f}, max {POS_WEIGHT.max():.2f}")


split     docs  statutes per doc   chars (median / 90%)
train   42,750              3.70       2,717 /   16,789
dev     10,181              3.92       3,418 /   17,900
test    13,019               nan       3,368 /   17,277

100 statutes (ids 0 .. 99) | train documents per statute: min 272, median 679, max 11730
positive weights: min 1.63, median 7.84, max 10.00


### CELL 6 · Tokenize + chunks

**In simple words:** Cut every document into up to 8 chunks of 512 tokens: the whole document if it fits, otherwise 8 chunks spread over its start, middle and end.

In [6]:
# ============================================================
# CELL 6 — TOKENIZE + CHUNKS
#
#   whole document fits in MAX_CHUNKS x CHUNK_LEN tokens -> its consecutive chunks
#   longer document -> MAX_CHUNKS chunks spread over it: the text is cut into
#   MAX_CHUNKS equal parts and the first CHUNK_LEN tokens of every part are kept
# ============================================================

MAX_TOKENS = MAX_CHUNKS * CHUNK_LEN


def make_chunks(text):
    """-> (list of token-id arrays, True if the document was too long to read completely)"""

    text = clean(text)

    if len(text) <= MAX_TOKENS * 4:                      # short enough to tokenize completely (~2.3 chars per token)

        ids = encode_fast(text) or encode_fast(".")      # (an empty document becomes ".")

        if len(ids) <= MAX_TOKENS:
            chunks = [ids[k:k + CHUNK_LEN] for k in range(0, len(ids), CHUNK_LEN)]
            return [np.array(c, dtype=np.int32) for c in chunks], False

    # long document: MAX_CHUNKS chunks spread over its start, middle and end
    step = len(text) / MAX_CHUNKS
    chunks = [head_tokens(text[int(k * step):int((k + 1) * step)], CHUNK_LEN) for k in range(MAX_CHUNKS)]

    return [np.array(c, dtype=np.int32) for c in chunks if len(c)], True


t0 = time.time()

for split, docs in [("train", train_docs), ("dev", dev_docs), ("test", test_docs)]:

    for k, d in enumerate(docs):
        d["chunks"], d["cut"] = make_chunks(d["text"])
        del d["text"]                                    # free memory (the text is not needed any more)

        if (k + 1) % 10000 == 0:
            print(f"  {split}: {k + 1}/{len(docs)} documents | {time.time() - t0:.0f}s", flush=True)

print(f"Tokenized in {time.time() - t0:.0f}s")

n_chunks = Counter(len(d["chunks"]) for d in train_docs)
print(f"\nChunks per train document: {dict(sorted(n_chunks.items()))}")
for split, docs in [("train", train_docs), ("dev", dev_docs), ("test", test_docs)]:
    print(f"{split:6s}: mean {np.mean([len(d['chunks']) for d in docs]):.2f} chunks per document | "
          f"read completely {1 - np.mean([d['cut'] for d in docs]):.1%} | "
          f"longer than {MAX_TOKENS} tokens ({MAX_CHUNKS} chunks spread over the document) {np.mean([d['cut'] for d in docs]):.1%}")

sample_ids = np.concatenate([c for d in train_docs[:2000] for c in d["chunks"]])
print(f"\nwhitespace tokens: {np.isin(sample_ids, SPACE_TOKEN_IDS).mean():.1%} of all tokens "
      f"({'not pooled' if POOL_SKIP_SPACES else 'pooled'})")


  train: 10000/42750 documents | 15s
  train: 20000/42750 documents | 28s
  train: 30000/42750 documents | 41s
  train: 40000/42750 documents | 54s
  dev: 10000/10181 documents | 73s
  test: 10000/13019 documents | 88s
Tokenized in 92s

Chunks per train document: {1: 11500, 2: 9303, 3: 4129, 4: 2541, 5: 1891, 6: 1558, 7: 1397, 8: 10431}
train : mean 3.85 chunks per document | read completely 78.5% | longer than 4096 tokens (8 chunks spread over the document) 21.5%
dev   : mean 4.16 chunks per document | read completely 75.3% | longer than 4096 tokens (8 chunks spread over the document) 24.7%
test  : mean 4.11 chunks per document | read completely 76.9% | longer than 4096 tokens (8 chunks spread over the document) 23.1%

whitespace tokens: 39.6% of all tokens (not pooled)


### CELL 7 · Metrics + thresholds (multi-label)

**In simple words:** The scores (macro-F1 = average over the statutes) and one threshold per statute, chosen on dev. A document with no statute above its threshold still gets its most likely one.

In [7]:
# ============================================================
# CELL 7 — METRICS + THRESHOLDS (multi-label)
#   macro-F1 (the competition metric: mean F1 over the statutes), micro-F1, subset accuracy
#   one threshold per statute = the one with the best F1 for that statute on dev
#   "at least one": a document with no statute above its threshold gets its most probable one
# ============================================================

def multilabel_scores(Y, P):
    """Y, P: [docs, statutes] 0/1 arrays."""

    Y, P = Y.astype(bool), P.astype(bool)
    tp = (Y & P).sum(axis=0)
    fp = (~Y & P).sum(axis=0)
    fn = (Y & ~P).sum(axis=0)

    f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)

    return {
        "macro_f1": float(f1.mean()),
        "micro_f1": float(2 * tp.sum() / max(2 * tp.sum() + fp.sum() + fn.sum(), 1)),
        "subset_acc": float((Y == P).all(axis=1).mean()),
        "macro_precision": float(np.mean(tp / np.maximum(tp + fp, 1))),
        "macro_recall": float(np.mean(tp / np.maximum(tp + fn, 1))),
        "per_statute_f1": f1,
    }


THRESHOLD_GRID = np.linspace(0.02, 0.98, 49)


def tune_thresholds(Y, probs):
    """One threshold per statute: the one with the best F1 on these documents."""

    Y = Y.astype(bool)
    best_f1 = np.full(Y.shape[1], -1.0)
    best_t = np.full(Y.shape[1], 0.5)

    for t in THRESHOLD_GRID:
        P = probs >= t
        tp = (Y & P).sum(axis=0)
        f1 = 2 * tp / np.maximum(P.sum(axis=0) + Y.sum(axis=0), 1)
        better = f1 > best_f1
        best_f1[better], best_t[better] = f1[better], t

    return best_t


def decide(probs, thresholds, at_least_one):
    P = probs >= thresholds
    if at_least_one:
        empty = ~P.any(axis=1)
        P[np.where(empty)[0], probs[empty].argmax(axis=1)] = True
    return P


def dev_scores(Y, probs):
    """Thresholds tuned on these documents; returns scores, thresholds, at_least_one."""

    thresholds = tune_thresholds(Y, probs)
    best = None

    for at_least_one in (False, True):
        s = multilabel_scores(Y, decide(probs, thresholds, at_least_one))
        if best is None or s["macro_f1"] > best[0]["macro_f1"]:
            best = (s, thresholds, at_least_one)

    return best


# check on a tiny example
_Y = np.array([[1, 0], [0, 1], [1, 1]])
_s = multilabel_scores(_Y, np.array([[1, 0], [0, 0], [1, 1]]))
assert abs(_s["macro_f1"] - (1.0 + 2 / 3) / 2) < 1e-9 and abs(_s["subset_acc"] - 2 / 3) < 1e-9
print("✅ metrics check passed")


✅ metrics check passed


### CELL 8 · Model

**In simple words:** Every chunk goes through the Phase-1 model; two parts then give 100 statute scores: a simple classifier on the average token vector, and *label attention*, where every statute looks for the words that signal it.

**How it works:** token vector = learned mix of all layers, standardized; whitespace tokens are not pooled. Linear path starts as a logistic regression; label attention `a_s = softmax(q_s · tanh(K z))` starts at 0. Adaptation: AdaLoRA: rank 12 → average 8, importance-based pruning of singular triplets.

In [8]:
# ============================================================
# CELL 8 — MODEL
#
#   every chunk -> Architecture-3 E-LegalFormer (Phase-1 weights) + AdaLoRA
#               -> token vector = learned mix of ALL layers (embeddings + every block)
#               -> token vectors standardized
#   tokens of a document (whitespace tokens and the first tokens of a chunk left out):
#     LINEAR path    : chunk vector = mean of its tokens, standardized;
#                      mean over the chunks -> Linear -> 100 statute scores
#                      (starts as the logistic regression of CELL 11)
#     LABEL ATTENTION: for every statute s, attention weights over all tokens
#                      a_s = softmax(q_s . tanh(K z)) -> v_s = sum a_s z -> score w_s . v_s
#                      (w starts at 0: the classifier starts exactly as the logistic regression)
#   scores = linear + label attention -> sigmoid -> P(statute applies)
#
# Layer names match the Phase-1 notebook, so the weights load directly.
# ============================================================

# ============================================================
# Architecture-3 E-LegalFormer (causal decoder, GQA, RoPE)
# ============================================================

class RMSNorm(nn.Module):

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        x_float = x.float()                                   # float32 for safety under float16
        x_float = x_float * torch.rsqrt(x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x_float * self.weight).to(x.dtype)


class RotaryEmbedding(nn.Module):

    def __init__(self, dim, max_seq_len, theta=10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        freqs = torch.outer(torch.arange(max_seq_len), inv_freq)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, seq_len):
        return self.cos[:seq_len], self.sin[:seq_len]


def rotate_half(x):
    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(q, k, cos, sin):
    cos = torch.cat([cos, cos], dim=-1)[None, None].to(q.dtype)
    sin = torch.cat([sin, sin], dim=-1)[None, None].to(q.dtype)
    return q * cos + rotate_half(q) * sin, k * cos + rotate_half(k) * sin


class GroupedQueryAttention(nn.Module):
    """12 query heads share 4 key/value heads. Causal, as in pretraining."""

    def __init__(self, cfg):
        super().__init__()
        self.n_q_heads = cfg["n_q_heads"]
        self.n_kv_heads = cfg["n_kv_heads"]
        self.head_dim = cfg["d_model"] // cfg["n_q_heads"]
        self.q_proj = nn.Linear(cfg["d_model"], self.n_q_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(cfg["d_model"], self.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg["d_model"], self.n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(cfg["d_model"], cfg["d_model"], bias=False)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_q_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, k, cos, sin)

        repeat = self.n_q_heads // self.n_kv_heads
        k = k.repeat_interleave(repeat, dim=1)
        v = v.repeat_interleave(repeat, dim=1)

        # causal; padding is at the END, so real tokens never see it
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(out)


class SwiGLU(nn.Module):

    def __init__(self, d_model, ffn_dim):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, ffn_dim, bias=False)
        self.up_proj = nn.Linear(d_model, ffn_dim, bias=False)
        self.down_proj = nn.Linear(ffn_dim, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class ELegalFormerBlock(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg["d_model"])
        self.attention = GroupedQueryAttention(cfg)
        self.ffn_norm = RMSNorm(cfg["d_model"])
        self.ffn = SwiGLU(cfg["d_model"], cfg["ffn_dim"])

    def forward(self, x, cos, sin):
        x = x + self.attention(self.attn_norm(x), cos, sin)
        x = x + self.ffn(self.ffn_norm(x))
        return x


class ELegalFormer(nn.Module):
    """Architecture-3 without the LM head: tokens -> hidden vectors."""

    def __init__(self, cfg):
        super().__init__()
        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.rope = RotaryEmbedding(cfg["d_model"] // cfg["n_q_heads"], cfg["max_seq_len"], cfg.get("rope_theta", 10000.0))
        self.layers = nn.ModuleList([ELegalFormerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = RMSNorm(cfg["d_model"])

    def forward(self, input_ids, mask=None, all_layers=False):
        """`mask` is not needed: attention is causal and padding is at the end.
        all_layers=False: final hidden vectors (as in Phase 1).
        all_layers=True : list of the embeddings + the output of every layer (before any norm)."""
        x = self.token_embedding(input_ids)
        cos, sin = self.rope(input_ids.size(1))
        states = [x]
        for layer in self.layers:
            x = layer(x, cos, sin)
            states.append(x)
        if all_layers:
            return states
        return self.final_norm(x)


def backbone_blocks(backbone):
    """The transformer blocks of the Phase-1 model, bottom -> top."""

    return list(backbone.layers)


# ------------------------------------------------------------
# AdaLoRA layer:  y = W x + alpha/r * P diag(E * mask) Q x
# ------------------------------------------------------------

class AdaLoRALinear(nn.Module):

    def __init__(self, base, rank, alpha, dropout):
        super().__init__()

        self.base = base                                          # frozen Phase-1 layer
        out_features, in_features = base.weight.shape

        self.lora_Q = nn.Parameter(0.02 * torch.randn(rank, in_features))
        self.lora_E = nn.Parameter(torch.zeros(rank))             # starts at 0 -> W' = W
        self.lora_P = nn.Parameter(0.02 * torch.randn(out_features, rank))
        self.register_buffer("rank_mask", torch.ones(rank))      # 0 = triplet switched off
        self.dropout = nn.Dropout(dropout)
        self.scale = alpha / rank

    def forward(self, x):

        y = self.base(x)
        h = self.dropout(x) @ self.lora_Q.T.to(x.dtype)                         # [.., r]
        h = h * (self.lora_E * self.rank_mask).to(x.dtype)

        return y + self.scale * (h @ self.lora_P.T.to(x.dtype))


def add_peft(backbone):
    """Freeze every Phase-1 weight, then wrap the target linear layers of every block with AdaLoRA."""

    for p in backbone.parameters():
        p.requires_grad = False

    for block in backbone_blocks(backbone):
        for target in ADALORA_TARGETS:

            parent_name, name = target.rsplit(".", 1)
            parent = block.get_submodule(parent_name)

            setattr(parent, name, AdaLoRALinear(getattr(parent, name), ADALORA_INIT_RANK, ADALORA_ALPHA, ADALORA_DROPOUT))


def adalora_layers(model):
    return [m for m in model.backbone.modules() if isinstance(m, AdaLoRALinear)]


def orthogonality_loss(model):
    """Mean over the AdaLoRA layers of ||P^T P - I||_F + ||Q Q^T - I||_F."""

    total, layers = 0.0, adalora_layers(model)

    for layer in layers:
        eye = torch.eye(ADALORA_INIT_RANK, device=layer.lora_P.device)
        total = total + torch.linalg.norm(layer.lora_P.T @ layer.lora_P - eye) \
                      + torch.linalg.norm(layer.lora_Q @ layer.lora_Q.T - eye)

    return ADALORA_ORTH_WEIGHT * total / len(layers)


class RankAllocator:
    """Importance of every (P_i, E_i, Q_i) triplet + the rank budget schedule."""

    def __init__(self, model, total_steps):

        self.layers = adalora_layers(model)
        self.total_steps = total_steps                        # updates in which AdaLoRA trains
        self.init_budget = ADALORA_INIT_RANK * len(self.layers)
        self.target_budget = ADALORA_TARGET_RANK * len(self.layers)
        self.smooth, self.uncertainty = {}, {}

    def budget(self, t):
        """Number of active triplets after t updates: init -> target (cubic), then fixed."""

        final_start = self.total_steps - ADALORA_T_FINAL
        if t < ADALORA_T_INIT:
            return self.init_budget
        if t >= final_start:
            return self.target_budget

        progress = (t - ADALORA_T_INIT) / max(1, final_start - ADALORA_T_INIT)
        return int(self.target_budget + (self.init_budget - self.target_budget) * (1 - progress) ** 3)

    @torch.no_grad()
    def update_importance(self):

        for layer in self.layers:
            for p in (layer.lora_P, layer.lora_E, layer.lora_Q):

                if p.grad is None:
                    continue

                s = (p * p.grad).abs()                                    # sensitivity |w x dL/dw|
                if p not in self.smooth:
                    self.smooth[p] = torch.zeros_like(s)
                    self.uncertainty[p] = torch.zeros_like(s)

                self.smooth[p].mul_(ADALORA_BETA).add_((1 - ADALORA_BETA) * s)
                self.uncertainty[p].mul_(ADALORA_BETA).add_((1 - ADALORA_BETA) * (s - self.smooth[p]).abs())

    def triplet_scores(self, layer):

        def score(p):
            return self.smooth[p] * self.uncertainty[p]

        return score(layer.lora_E) + score(layer.lora_Q).mean(dim=1) + score(layer.lora_P).mean(dim=0)

    @torch.no_grad()
    def step(self, t):
        """Called after every update in which AdaLoRA trained (t = such updates so far)."""

        self.update_importance()

        final_start = self.total_steps - ADALORA_T_FINAL
        if t < ADALORA_T_INIT or t > final_start or (t % ADALORA_DELTA_T != 0 and t != final_start):
            return
        if any(layer.lora_E not in self.smooth for layer in self.layers):
            return

        scores = [self.triplet_scores(layer) for layer in self.layers]
        all_scores = torch.cat(scores)
        keep = self.budget(t)

        if keep >= len(all_scores):
            return

        threshold = torch.kthvalue(all_scores.float().cpu(), len(all_scores) - keep).values.to(all_scores.device)
        for layer, s in zip(self.layers, scores):
            layer.rank_mask.copy_((s > threshold).float())

    def active(self):
        return int(sum(layer.rank_mask.sum().item() for layer in self.layers))


def rank_summary(model):
    """Active rank per layer type (mean over the blocks)."""

    summary = {}
    for target in ADALORA_TARGETS:
        ranks = [int(block.get_submodule(target).rank_mask.sum().item()) for block in backbone_blocks(model.backbone)]
        summary[target] = {"mean": float(np.mean(ranks)), "per_block": ranks}
    return summary


def backbone_groups(model):
    """(parameters, learning rate, weight decay) of the trained Phase-1 part."""
    return [(model.backbone_parameters(), LR_PEFT, 0.0)]


def check_trainable(backbone):
    names = [name for name, p in backbone.named_parameters() if p.requires_grad]
    assert names and all("lora_" in name for name in names), "only the AdaLoRA matrices may train in the Phase-1 model"


# ============================================================
# Standardization with RUNNING statistics (no learned weights)
#   x -> (x - mean) / std
# The statistics start from the frozen Phase-1 model (CELL 11) and then
# slowly follow the features while training changes them.
# ============================================================

class RunningStandardize(nn.Module):

    def __init__(self, dim, momentum=0.01):
        super().__init__()
        self.momentum = momentum
        self.register_buffer("mean", torch.zeros(dim))
        self.register_buffer("std", torch.ones(dim))

    def forward(self, x, mask):
        """x [..., dim], mask [...] (True = real value)."""

        if self.training:
            with torch.no_grad():
                values = x[mask]
                if len(values) > 1:
                    self.mean.mul_(1 - self.momentum).add_(self.momentum * values.mean(dim=0))
                    self.std.mul_(1 - self.momentum).add_(self.momentum * values.std(dim=0).clamp(min=1e-4))

        return (x - self.mean) / self.std


# ============================================================
# Chunks -> padded token ids, and which tokens are pooled
# ============================================================

SPACE_IDS = torch.tensor(SPACE_TOKEN_IDS if POOL_SKIP_SPACES and SPACE_TOKEN_IDS else [-1], dtype=torch.long, device=DEVICE)


def pad_chunks(chunks):
    """List of token-id arrays -> ids [n, T], mask [n, T] (True = real token), on the GPU."""

    max_len = max(len(c) for c in chunks)
    ids = torch.full((len(chunks), max_len), PAD_ID, dtype=torch.long)
    mask = torch.zeros((len(chunks), max_len), dtype=torch.bool)

    for row, c in enumerate(chunks):
        ids[row, :len(c)] = torch.from_numpy(c.astype(np.int64))
        mask[row, :len(c)] = True

    return ids.to(DEVICE), mask.to(DEVICE)


def pooling_mask(ids, mask):
    """Tokens that are pooled: real tokens, not whitespace, not the first POOL_SKIP_TOKENS of a chunk."""

    pool = mask & ~torch.isin(ids, SPACE_IDS)

    long_chunk = pool.sum(dim=1) > 2 * POOL_SKIP_TOKENS
    pool[long_chunk, :POOL_SKIP_TOKENS] = False

    empty = ~pool.any(dim=1)                   # a chunk with nothing left keeps all its real tokens
    pool[empty] = mask[empty]

    return pool


# ============================================================
# Document classifier (multi-label)
# ============================================================

class DocumentClassifier(nn.Module):

    def __init__(self, backbone, d_model, n_states):
        super().__init__()

        self.backbone = backbone
        self.d_model = d_model
        self.train_backbone = True          # False = the Phase-1 model runs without gradients (head-only epoch)

        # layer mix: token vector = sum_l w_l * LayerNorm(h_l), w = softmax(layer_logits)
        # (start: CELL 11, the best layers get the largest weights)
        self.layer_logits = nn.Parameter(torch.zeros(n_states))

        self.token_standardize = RunningStandardize(d_model)
        self.chunk_standardize = RunningStandardize(d_model)

        # LINEAR path: starts as the logistic regression of CELL 11
        self.linear_path = nn.Linear(d_model, NUM_LABELS)

        # LABEL ATTENTION: one query and one output vector per statute (output starts at 0)
        self.label_key = nn.Linear(d_model, LABEL_ATTENTION_DIM)
        self.label_query = nn.Parameter(0.02 * torch.randn(NUM_LABELS, LABEL_ATTENTION_DIM))
        self.label_output = nn.Parameter(torch.zeros(NUM_LABELS, d_model))
        self.dropout = nn.Dropout(HEAD_DROPOUT)

        self.register_buffer("pos_weight", POS_WEIGHT.clone())

    def backbone_parameters(self):
        return [p for p in self.backbone.parameters() if p.requires_grad]

    def head_parameters(self):
        return [p for name, p in self.named_parameters() if not name.startswith("backbone.")]

    def layer_weights(self):
        return torch.softmax(self.layer_logits.float(), dim=0)

    def mixed_states(self, ids, mask):
        """[n, T] ids -> [n, T, D] float32: mix of the LayerNorm-ed hidden states of all layers."""

        with torch.set_grad_enabled(torch.is_grad_enabled() and self.train_backbone):
            with autocast():
                states = self.backbone(ids, mask, all_layers=True)

        mixed = 0.0
        for w, h in zip(self.layer_weights(), states):
            mixed = mixed + w * F.layer_norm(h.float(), h.shape[-1:])

        return mixed

    def forward(self, docs):
        """List of documents -> statute scores [n_docs, NUM_LABELS] (logits)."""

        chunks = [c for d in docs for c in d["chunks"]]
        ids, mask = pad_chunks(chunks)
        pool = pooling_mask(ids, mask)

        z = self.token_standardize(self.mixed_states(ids, mask), pool)             # [n_chunks, T, D]

        # chunk vector = mean of its pooled token vectors, standardized
        m = pool.unsqueeze(-1).float()
        chunk_vectors = self.chunk_standardize((z * m).sum(dim=1) / m.sum(dim=1),
                                               torch.ones(len(chunks), dtype=torch.bool, device=DEVICE))

        scores, k = [], 0

        for d in docs:
            n = len(d["chunks"])

            # LINEAR path: mean over the chunks of the document
            linear = self.linear_path(chunk_vectors[k:k + n].mean(dim=0))

            # LABEL ATTENTION over all pooled tokens of the document
            tokens = z[k:k + n][pool[k:k + n]]                                       # [N, D]
            keys = torch.tanh(self.label_key(tokens))                                # [N, A]
            attention = torch.softmax(keys @ self.label_query.T, dim=0)             # [N, statutes], sums to 1 over the tokens
            per_statute = self.dropout(attention.T @ tokens)                         # [statutes, D]

            scores.append(linear + (per_statute * self.label_output).sum(dim=-1))
            k += n

        return torch.stack(scores)

    def loss(self, docs):
        logits = self(docs).float()
        y = torch.tensor(np.stack([d["y"] for d in docs]), device=DEVICE)
        return F.binary_cross_entropy_with_logits(logits, y, pos_weight=self.pos_weight)


### CELL 9 · Build the model from the Phase-1 Architecture-3 checkpoint

**In simple words:** Create the network, load the trained Phase-1 weights into it, and add the adaptation (the part that will learn).

**How it works:** The layer names match the Phase-1 notebook, so the checkpoint loads directly; if it is not at the configured path, the notebook searches `/kaggle/input`. The printed counts show how many weights train. Here: AdaLoRA: rank 12 → average 8, importance-based pruning of singular triplets.

In [9]:
# ============================================================
# CELL 9 — BUILD THE MODEL FROM THE PHASE-1 ARCHITECTURE-3 CHECKPOINT
# ============================================================

def is_architecture3(path):

    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        return isinstance(ckpt, dict) and "n_kv_heads" in ckpt.get("config", {})
    except Exception:
        return False


if not CHECKPOINT_PATH.exists():

    found = [p for p in sorted(Path("/kaggle/input").rglob("best_model.pt")) if is_architecture3(p)]
    assert found, f"{CHECKPOINT_PATH} not found and no Architecture-3 checkpoint under /kaggle/input"

    CHECKPOINT_PATH = found[0]
    print("Checkpoint found at:", CHECKPOINT_PATH)


phase1 = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
MODEL_CONFIG = dict(phase1["config"])

# weights without the LM head (tied to the embedding, not needed here)
MODEL_WEIGHTS = {
    key.replace("_orig_mod.", ""): value
    for key, value in phase1["model_state_dict"].items()
    if not key.replace("_orig_mod.", "").startswith("lm_head.")
}

del phase1

D_MODEL = MODEL_CONFIG["d_model"]

print("Phase-1 config:", MODEL_CONFIG)

assert MODEL_CONFIG["vocab_size"] == VOCAB_SIZE, "checkpoint was trained with a different tokenizer"
assert CHUNK_LEN <= MODEL_CONFIG["max_seq_len"], f"CHUNK_LEN must be <= {MODEL_CONFIG['max_seq_len']}"


def build_model():
    """Phase-1 model + AdaLoRA + new document classifier."""

    backbone = ELegalFormer(MODEL_CONFIG)
    backbone.load_state_dict(MODEL_WEIGHTS, strict=True)          # every name and shape must match

    test_ids = torch.randint(4, VOCAB_SIZE, (2, 24))
    test_mask = torch.ones(2, 24, dtype=torch.bool)

    backbone.eval()
    with torch.no_grad():
        before = backbone(test_ids, test_mask)
        add_peft(backbone)
        after = backbone(test_ids, test_mask)
        states = backbone(test_ids, test_mask, all_layers=True)
    backbone.train()

    assert torch.allclose(before, after, atol=1e-4), "AdaLoRA changed the model at the start"
    assert len(states) == len(backbone_blocks(backbone)) + 1
    assert torch.allclose(backbone.final_norm(states[-1]), after, atol=1e-4)
    check_trainable(backbone)

    return DocumentClassifier(backbone, D_MODEL, len(backbone_blocks(backbone)) + 1).to(DEVICE)


for name in ["model", "optimizer", "scaler"]:
    globals().pop(name, None)
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

model = build_model()

N_BLOCKS = len(backbone_blocks(model.backbone))

n_total = sum(p.numel() for p in model.parameters())
n_backbone = sum(p.numel() for p in model.backbone.parameters())
n_trained = sum(p.numel() for p in model.backbone_parameters())
n_head = sum(p.numel() for p in model.head_parameters())

print(f"\nPhase-1 model             : {n_backbone:,} ({N_BLOCKS} blocks)")
print(f"trained in it (AdaLoRA): {n_trained:,}  ({n_trained / n_backbone:.2%})")
print(f"head (trainable)          : {n_head:,}")


Phase-1 config: {'vocab_size': 32004, 'max_seq_len': 2048, 'd_model': 768, 'n_layers': 16, 'n_q_heads': 12, 'n_kv_heads': 4, 'ffn_dim': 2048, 'dropout': 0.0, 'rope_theta': 10000.0}

Phase-1 model             : 127,874,112 (16 blocks)
trained in it (AdaLoRA): 2,606,400  (2.04%)
head (trainable)          : 376,181


### CELL 10 · Prediction + evaluation

**In simple words:** Predict the 100 statute probabilities of documents (in groups that fit the GPU) and score them.

In [10]:
# ============================================================
# CELL 10 — PREDICTION + EVALUATION
#   documents are processed in groups of up to EVAL_CHUNKS chunks
# ============================================================

@torch.no_grad()
def predict_probs(model, docs):
    """[n_docs, NUM_LABELS] probabilities."""

    model.eval()

    probs, group, n_chunks = [], [], 0

    for d in docs + [None]:

        if group and (d is None or n_chunks + len(d["chunks"]) > EVAL_CHUNKS):
            probs.append(torch.sigmoid(model(group).float()).cpu())
            group, n_chunks = [], 0

        if d is not None:
            group.append(d)
            n_chunks += len(d["chunks"])

    model.train()

    return torch.cat(probs).numpy()


def evaluate(model, docs):
    """Macro-F1 etc. with thresholds tuned on these documents (used to pick the best epoch)."""

    Y = np.stack([d["y"] for d in docs])
    probs = predict_probs(model, docs)
    scores, thresholds, at_least_one = dev_scores(Y, probs)

    return scores, probs


### CELL 11 · Before training: probe every layer + start of the classifier

**In simple words:** Before training: find which layers know most about statutes, store feature statistics, and start the classifier from a logistic regression — so training starts from a sensible model, not a random one.

**How it works:** A linear probe on every layer gives dev macro-F1 → the layer mix starts on the best layers; the linear path is fitted with the same loss as training (BCE with positive weights).

In [11]:
# ============================================================
# CELL 11 — BEFORE TRAINING: PROBE EVERY LAYER + START OF THE CLASSIFIER
#
#   1. per-layer means (LayerNorm-ed token vectors, averaged over the chunks) of
#      PROBE_DOCS train documents and DEV_EVAL_DOCS dev documents (frozen Phase-1 model)
#   2. probe: a linear multi-label model on every layer -> dev macro-F1
#      -> the layer mix STARTS on the best layers
#   3. token and chunk statistics of the mixed vectors -> stored in the model
#   4. the LINEAR path starts as a logistic regression (same loss as training:
#      BCE with the positive weights) on exactly the features it sees
# ============================================================

t0 = time.time()

N_STATES = len(model.layer_logits)
LAYER_NAMES = ["embeddings"] + [f"layer {i}" for i in range(1, N_STATES)]


def chunk_batches(chunks):
    """Batches of EVAL_CHUNKS chunks: (ids, mask, pooling mask) on the GPU."""

    for k in range(0, len(chunks), EVAL_CHUNKS):
        ids, mask = pad_chunks(chunks[k:k + EVAL_CHUNKS])
        yield ids, mask, pooling_mask(ids, mask)


@torch.no_grad()
def layer_means(docs):
    """[n_docs, N_STATES, D]: per document and layer, the mean LayerNorm-ed token vector (averaged over its chunks)."""

    model.eval()

    chunks = [c for d in docs for c in d["chunks"]]
    per_chunk = []

    for ids, mask, pool in chunk_batches(chunks):
        with autocast():
            states = model.backbone(ids, mask, all_layers=True)
        m = pool.unsqueeze(-1)
        per_chunk.append(torch.stack(
            [(F.layer_norm(h.float(), h.shape[-1:]) * m).sum(dim=1) / m.sum(dim=1) for h in states], dim=1).cpu())

    model.train()

    per_chunk = torch.cat(per_chunk)
    out, k = [], 0
    for d in docs:
        n = len(d["chunks"])
        out.append(per_chunk[k:k + n].mean(dim=0))
        k += n

    return torch.stack(out)


def fit_linear(X, Y, weight_decay, steps=300):
    """Multi-label logistic regression on fixed features X [n, D] (full batch, Adam), with the
    training loss (BCE + positive weights) + L2. Returns W [D, L], b [L]."""

    X = torch.as_tensor(X, dtype=torch.float32, device=DEVICE)
    Y = torch.as_tensor(Y, dtype=torch.float32, device=DEVICE)
    W = torch.zeros(X.shape[1], Y.shape[1], device=DEVICE, requires_grad=True)
    b = torch.logit(Y.mean(dim=0).clamp(1e-3, 1 - 1e-3)).clone().requires_grad_(True)

    optimizer = torch.optim.Adam([W, b], lr=0.01)
    pos_weight = POS_WEIGHT.to(DEVICE)

    with torch.enable_grad():
        for _ in range(steps):
            loss = F.binary_cross_entropy_with_logits(X @ W + b, Y, pos_weight=pos_weight) + weight_decay * W.pow(2).sum()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    return W.detach(), b.detach()


def linear_probs(X, W, b):
    return torch.sigmoid(torch.as_tensor(X, dtype=torch.float32, device=DEVICE) @ W + b).cpu().numpy()


# ---------------- 1. per-layer means ----------------
probe_docs = random.Random(SEED + 1).sample(train_docs, min(PROBE_DOCS, len(train_docs)))
dev_eval_docs = random.Random(SEED + 3).sample(dev_docs, min(DEV_EVAL_DOCS, len(dev_docs)))

L_probe = layer_means(probe_docs)
L_dev = layer_means(dev_eval_docs)
Y_probe = np.stack([d["y"] for d in probe_docs])
Y_dev_eval = np.stack([d["y"] for d in dev_eval_docs])

print(f"per-layer means of {len(probe_docs)} train + {len(dev_eval_docs)} dev documents | {time.time() - t0:.0f}s")


# ---------------- 2. probe every layer ----------------
print(f"\n  {'layer':12s} {'dev macro-F1':>13s} {'micro-F1':>9s}")
probe_results = {}

for l, name in enumerate(LAYER_NAMES):

    mu = L_probe[:, l].mean(dim=0)
    sd = L_probe[:, l].std(dim=0).clamp(min=1e-6)
    W, b = fit_linear((L_probe[:, l] - mu) / sd, Y_probe, weight_decay=1e-4, steps=200)
    scores, _, _ = dev_scores(Y_dev_eval, linear_probs((L_dev[:, l] - mu) / sd, W, b))

    probe_results[name] = {"macro_f1": scores["macro_f1"], "micro_f1": scores["micro_f1"]}
    print(f"  {name:12s} {scores['macro_f1']:13.4f} {scores['micro_f1']:9.4f}")

PROBE_BEST_LAYER = max(probe_results, key=lambda n: probe_results[n]["macro_f1"])
PROBE_BEST = probe_results[PROBE_BEST_LAYER]

# the layer mix starts on the best layers: weight ~ exp((F1 - best F1) / 0.02), so layers that
# are almost as good share the weight and much worse layers start near 0 (the mix is learned)
with torch.no_grad():
    model.layer_logits.copy_(torch.tensor(
        [(probe_results[n]["macro_f1"] - PROBE_BEST["macro_f1"]) / 0.02 for n in LAYER_NAMES]).to(DEVICE))

start_mix = model.layer_weights().detach().cpu()
print(f"\nbest frozen layer: {PROBE_BEST_LAYER} (dev macro-F1 {PROBE_BEST['macro_f1']:.4f})")
print("layer mix at the start:", {n: round(float(w), 3) for n, w in zip(LAYER_NAMES, start_mix)})


# ---------------- 3. token + chunk statistics of the mixed vectors ----------------
stat_docs = random.Random(SEED).sample(train_docs, min(STATS_DOCS, len(train_docs)))
stat_chunks = [c for d in stat_docs for c in d["chunks"]]

total = torch.zeros(D_MODEL, dtype=torch.float64, device=DEVICE)
total_sq = torch.zeros_like(total)
count = 0

model.eval()
with torch.no_grad():

    for ids, mask, pool in chunk_batches(stat_chunks):
        hidden = model.mixed_states(ids, mask)
        m = pool.unsqueeze(-1)
        total += (hidden * m).sum(dim=(0, 1)).double()
        total_sq += (hidden.pow(2) * m).sum(dim=(0, 1)).double()
        count += int(pool.sum())

    TOKEN_MEAN = (total / count).float()
    TOKEN_STD = (total_sq / count - (total / count).pow(2)).clamp(min=1e-8).sqrt().float().clamp(min=1e-4)
    model.token_standardize.mean.copy_(TOKEN_MEAN)
    model.token_standardize.std.copy_(TOKEN_STD)

    chunk_means = []
    for ids, mask, pool in chunk_batches(stat_chunks):
        z = model.token_standardize(model.mixed_states(ids, mask), pool)
        m = pool.unsqueeze(-1).float()
        chunk_means.append(((z * m).sum(dim=1) / m.sum(dim=1)).cpu())

    chunk_means = torch.cat(chunk_means)
    CHUNK_MEAN = chunk_means.mean(dim=0)
    CHUNK_STD = chunk_means.std(dim=0).clamp(min=1e-4)
    model.chunk_standardize.mean.copy_(CHUNK_MEAN.to(DEVICE))
    model.chunk_standardize.std.copy_(CHUNK_STD.to(DEVICE))

model.train()

print(f"\nfeature statistics from {count:,} pooled tokens of {len(stat_docs)} train documents")


# ---------------- 4. the linear path starts as a logistic regression ----------------
# document feature = mean over its chunks of the standardized chunk means of the MIXED token
# vectors (exactly what the linear path sees; everything is linear, so it follows from the
# per-layer means of step 1)

def linear_path_features(L):
    mixed = (L * start_mix[None, :, None]).sum(dim=1)
    z = (mixed - TOKEN_MEAN.cpu()) / TOKEN_STD.cpu()
    return (z - CHUNK_MEAN) / CHUNK_STD


F_probe, F_dev = linear_path_features(L_probe), linear_path_features(L_dev)

best = None
for weight_decay in (1e-6, 1e-5, 1e-4, 1e-3):
    W, b = fit_linear(F_probe, Y_probe, weight_decay)
    scores, _, _ = dev_scores(Y_dev_eval, linear_probs(F_dev, W, b))
    print(f"  logistic regression (L2 {weight_decay:g}): dev macro-F1 {scores['macro_f1']:.4f}, micro-F1 {scores['micro_f1']:.4f}")
    if best is None or scores["macro_f1"] > best[0]["macro_f1"]:
        best = (scores, W, b)

START, W, b = best

with torch.no_grad():
    model.linear_path.weight.copy_(W.T)
    model.linear_path.bias.copy_(b)

print(f"\nThe classifier STARTS as this logistic regression: dev macro-F1 {START['macro_f1']:.4f} "
      f"(TF-IDF reference: 0.394) | CELL 11 took {time.time() - t0:.0f}s")

del L_probe, L_dev, F_probe, F_dev, chunk_means


per-layer means of 4000 train + 3000 dev documents | 416s

  layer         dev macro-F1  micro-F1
  embeddings          0.2483    0.3156
  layer 1             0.2209    0.3076
  layer 2             0.2357    0.3219
  layer 3             0.2418    0.3272
  layer 4             0.2440    0.3293
  layer 5             0.2474    0.3267
  layer 6             0.2523    0.3309
  layer 7             0.2518    0.3272
  layer 8             0.2496    0.3303
  layer 9             0.2496    0.3265
  layer 10            0.2507    0.3294
  layer 11            0.2508    0.3326
  layer 12            0.2521    0.3257
  layer 13            0.2492    0.3317
  layer 14            0.2496    0.3214
  layer 15            0.2514    0.3305
  layer 16            0.2508    0.3349

best frozen layer: layer 6 (dev macro-F1 0.2523)

feature statistics from 888,473 pooled tokens of 800 train documents
  logistic regression (L2 1e-06): dev macro-F1 0.2300, micro-F1 0.3191
  logistic regression (L2 1e-05): dev macro-F1 0

### CELL 12 · Optimizer + training function

**In simple words:** Optimizer and training loop: the first epochs train the head only, then the adapted Phase-1 weights join.

**How it works:** Epoch 0 = the starting classifier (kept if no epoch beats it). Loss = binary cross-entropy with a positive weight per statute (rare statutes count more).

In [12]:
# ============================================================
# CELL 12 — OPTIMIZER + TRAINING FUNCTION
#   groups: the trained Phase-1 part (AdaLoRA), then the head (last group)
#   epoch 0 = the start (kept if no epoch beats it), head-only epochs, then AdaLoRA matrices + head
# ============================================================

def make_optimizer(model, total_steps, head_only_steps):

    groups = [{"params": params, "lr": lr, "weight_decay": wd} for params, lr, wd in backbone_groups(model)]
    groups.append({"params": model.head_parameters(), "lr": LR_HEAD, "weight_decay": 0.01})

    optimizer = torch.optim.AdamW(groups, betas=(0.9, 0.999))

    base_lrs = [group["lr"] for group in optimizer.param_groups]
    warmup_steps = max(1, int(WARMUP_RATIO * total_steps))

    def set_lr(step):
        """Linear warmup, then linear decay to 0. The Phase-1 part waits for head_only_steps, then has its own warmup."""

        decay = 1.0 if step < warmup_steps else max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))

        head_factor = min(1.0, (step + 1) / warmup_steps) * decay
        backbone_factor = min(1.0, max(0, step + 1 - head_only_steps) / warmup_steps) * decay

        for group, base_lr in zip(optimizer.param_groups[:-1], base_lrs[:-1]):
            group["lr"] = base_lr * backbone_factor
        optimizer.param_groups[-1]["lr"] = base_lrs[-1] * head_factor

    return optimizer, set_lr


def trainable_state(model):
    """The trained weights + the running feature statistics (to go back to the best epoch)."""

    state = {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}
    state.update({name: b.detach().cpu().clone() for name, b in model.named_buffers()
                  if "standardize" in name or "rank_mask" in name})       # + which AdaLoRA triplets are on

    return state


def train_model(model, epochs):
    """
    Trains on TRAIN documents only. Every epoch uses EPOCH_DOCS documents (the train set is
    shuffled once and walked through). After every epoch: macro-F1 on DEV_EVAL_DOCS dev
    documents (thresholds tuned on them); the best epoch is kept.
    """

    docs_per_epoch = min(EPOCH_DOCS, len(train_docs))
    updates_per_epoch = math.ceil(docs_per_epoch / (BATCH_DOCS * GRAD_ACCUM))

    optimizer, set_lr = make_optimizer(model, epochs * updates_per_epoch, HEAD_ONLY_EPOCHS * updates_per_epoch)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)

    # AdaLoRA: importance of the triplets + rank budget over the updates in which AdaLoRA trains
    head_only_steps = HEAD_ONLY_EPOCHS * updates_per_epoch
    allocator = RankAllocator(model, epochs * updates_per_epoch - head_only_steps)

    backbone_params = [p for group in optimizer.param_groups[:-1] for p in group["params"]]
    head_params = optimizer.param_groups[-1]["params"]

    rng = random.Random(SEED)
    order = list(range(len(train_docs)))
    rng.shuffle(order)
    cursor = 0

    history = []
    step = 0
    t_start = time.time()

    # epoch 0 = the start (logistic regression on the frozen features): kept if no epoch beats it
    scores, _ = evaluate(model, dev_eval_docs)
    best_score, best_epoch, best_state = scores["macro_f1"], 0, trainable_state(model)
    history.append({"epoch": 0, "dev": {k: v for k, v in scores.items() if k != "per_statute_f1"}})
    print(f"Epoch 0 (start) | dev macro-F1 {scores['macro_f1']:.4f} | micro-F1 {scores['micro_f1']:.4f} | "
          f"subset acc {scores['subset_acc']:.4f}", flush=True)

    model.train()

    for epoch in range(1, epochs + 1):

        if cursor + docs_per_epoch > len(order):
            rng.shuffle(order)
            cursor = 0

        epoch_docs = [train_docs[i] for i in order[cursor:cursor + docs_per_epoch]]
        cursor += docs_per_epoch

        # head-only epochs: the Phase-1 model runs without gradients (AdaLoRA matrices do not train yet)
        model.train_backbone = epoch > HEAD_ONLY_EPOCHS

        batches = [epoch_docs[k:k + BATCH_DOCS] for k in range(0, len(epoch_docs), BATCH_DOCS)]

        total_loss, skipped = 0.0, 0
        optimizer.zero_grad(set_to_none=True)

        for b, batch in enumerate(batches):

            loss = model.loss(batch)
            total_loss += loss.item()

            if model.train_backbone:
                loss = loss + orthogonality_loss(model)            # AdaLoRA: keeps P and Q orthogonal

            scaler.scale(loss / GRAD_ACCUM).backward()

            if (b + 1) % GRAD_ACCUM == 0 or b + 1 == len(batches):

                set_lr(step)
                scaler.unscale_(optimizer)

                # clip the Phase-1 part and the head separately
                torch.nn.utils.clip_grad_norm_(backbone_params, GRAD_CLIP)
                torch.nn.utils.clip_grad_norm_(head_params, GRAD_CLIP)

                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                skipped += scaler.get_scale() < scale_before

                if model.train_backbone and scaler.get_scale() >= scale_before:
                    allocator.step(step - head_only_steps)          # AdaLoRA: importance + switch off triplets

                optimizer.zero_grad(set_to_none=True)
                step += 1

            if (b + 1) % 1000 == 0:
                print(f"    batch {b + 1}/{len(batches)} | loss {total_loss / (b + 1):.4f} | "
                      f"{(time.time() - t_start) / 60:.1f} min", flush=True)

        scores, _ = evaluate(model, dev_eval_docs)
        history.append({"epoch": epoch, "loss": total_loss / len(batches),
                        "dev": {k: v for k, v in scores.items() if k != "per_statute_f1"}, "skipped_steps": int(skipped)})

        line = (f"Epoch {epoch}/{epochs}{' (head only)' if epoch <= HEAD_ONLY_EPOCHS else ''} | loss {total_loss / len(batches):.4f} | "
                f"dev macro-F1 {scores['macro_f1']:.4f} | micro-F1 {scores['micro_f1']:.4f} | subset acc {scores['subset_acc']:.4f}")

        stop = False
        if scores["macro_f1"] > best_score:
            best_score, best_epoch, best_state = scores["macro_f1"], epoch, trainable_state(model)
            line += "  <- best"
        elif epoch - best_epoch >= PATIENCE:
            stop = True

        if skipped:
            line += f" | skipped {skipped}"
        print(line + f" | {(time.time() - t_start) / 60:.1f} min", flush=True)

        print(f"    AdaLoRA: {allocator.active()} of {allocator.init_budget} triplets on "
              f"(target {allocator.target_budget}, average rank {allocator.active() / len(allocator.layers):.1f})")

        mix = " ".join(f"{w:.2f}" for w in model.layer_weights().detach().tolist())
        print(f"    layer mix (embeddings, layer 1..): {mix}")

        if stop:
            print(f"Early stop: no improvement for {PATIENCE} epochs.")
            break

    model.load_state_dict(best_state, strict=False)          # back to the best epoch
    model.train_backbone = True

    return history, best_epoch


### CELL 13 · Train (train data only; dev picks the best epoch)

**In simple words:** Run the training: train data only; dev picks the best epoch.

In [13]:
# ============================================================
# CELL 13 — TRAIN (TRAIN data only; dev picks the best epoch)
# ============================================================

print(f"{RUN_NAME} | train {len(train_docs)} documents ({min(EPOCH_DOCS, len(train_docs))} per epoch) "
      f"-> dev {len(dev_eval_docs)} documents after every epoch")

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

t0 = time.time()
history, best_epoch = train_model(model, EPOCHS)
train_minutes = (time.time() - t0) / 60

peak_gpu_gb = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else 0.0

torch.save(trainable_state(model), OUTPUT_DIR / "best_trainable.pt")

print(f"\n✅ Trained in {train_minutes:.1f} min | best epoch {best_epoch} | peak GPU {peak_gpu_gb:.2f} GB")


p5_arch3_adalora | train 42750 documents (10000 per epoch) -> dev 3000 documents after every epoch
Epoch 0 (start) | dev macro-F1 0.2553 | micro-F1 0.3292 | subset acc 0.0013
    batch 1000/5000 | loss 0.3878 | 5.6 min
    batch 2000/5000 | loss 0.3913 | 8.1 min
    batch 3000/5000 | loss 0.3858 | 10.5 min
    batch 4000/5000 | loss 0.3837 | 12.9 min
    batch 5000/5000 | loss 0.3799 | 15.4 min
Epoch 1/4 (head only) | loss 0.3799 | dev macro-F1 0.2613 | micro-F1 0.3332 | subset acc 0.0020  <- best | 18.6 min
    AdaLoRA: 1344 of 1344 triplets on (target 896, average rank 12.0)
    layer mix (embeddings, layer 1..): 0.08 0.02 0.04 0.05 0.06 0.06 0.07 0.07 0.06 0.06 0.06 0.06 0.06 0.05 0.06 0.07 0.07
    batch 1000/5000 | loss 0.3580 | 25.4 min
    batch 2000/5000 | loss 0.3604 | 32.4 min
    batch 3000/5000 | loss 0.3597 | 39.2 min
    batch 4000/5000 | loss 0.3607 | 46.0 min
    batch 5000/5000 | loss 0.3587 | 52.8 min
Epoch 2/4 | loss 0.3587 | dev macro-F1 0.2806 | micro-F1 0.3608 | s

### CELL 14 · Dev report (all dev documents)

**In simple words:** Dev results: thresholds tuned on all dev documents (these are used for test), scores, F1 by how frequent a statute is, and the most missed / most over-predicted statutes.

In [14]:
# ============================================================
# CELL 14 — DEV REPORT (all dev documents)
#   thresholds per statute tuned on the full dev set (used for the test predictions),
#   scores, F1 by statute frequency, most missed / most over-predicted statutes
# ============================================================

t0 = time.time()
dev_probs = predict_probs(model, dev_docs)
predict_seconds = time.time() - t0

Y_dev = np.stack([d["y"] for d in dev_docs])

DEV_SCORES, THRESHOLDS, AT_LEAST_ONE = dev_scores(Y_dev, dev_probs)
dev_pred = decide(dev_probs, THRESHOLDS, AT_LEAST_ONE)

at_05 = multilabel_scores(Y_dev, decide(dev_probs, np.full(NUM_LABELS, 0.5), False))

print(f"DEV ({RUN_NAME}, best epoch {best_epoch}, {len(dev_docs)} documents, predicted in {predict_seconds:.0f}s)")
print(f"{'decision':38s} {'macro-F1':>9s} {'micro-F1':>9s} {'subset acc':>11s} {'macro-P':>8s} {'macro-R':>8s}")
print(f"{'threshold 0.5 for every statute':38s} {at_05['macro_f1']:9.4f} {at_05['micro_f1']:9.4f} {at_05['subset_acc']:11.4f} "
      f"{at_05['macro_precision']:8.4f} {at_05['macro_recall']:8.4f}")
print(f"{'threshold per statute' + (' + at least one' if AT_LEAST_ONE else ''):38s} {DEV_SCORES['macro_f1']:9.4f} "
      f"{DEV_SCORES['micro_f1']:9.4f} {DEV_SCORES['subset_acc']:11.4f} {DEV_SCORES['macro_precision']:8.4f} {DEV_SCORES['macro_recall']:8.4f}")
print(f"reference: start (logistic regression) {START['macro_f1']:.4f} on {len(dev_eval_docs)} dev docs | "
      f"TF-IDF word counts 0.394")

print(f"\nstatutes per document: gold {Y_dev.sum(axis=1).mean():.2f} | predicted {dev_pred.sum(axis=1).mean():.2f}")


# ------------------------------------------------------------
# F1 by how often the statute occurs in train
# ------------------------------------------------------------

per_f1 = DEV_SCORES["per_statute_f1"]
print(f"\n{'train docs per statute':>23s} {'statutes':>9s} {'mean F1':>8s}")
by_frequency = {}
for lo, hi in [(0, 499), (500, 999), (1000, 2999), (3000, 10 ** 9)]:
    group = [j for j in range(NUM_LABELS) if lo <= POSITIVES[j] <= hi]
    if group:
        name = f"{lo}-{hi}" if hi < 10 ** 9 else f">= {lo}"
        by_frequency[name] = {"statutes": len(group), "mean_f1": float(per_f1[group].mean())}
        print(f"{name:>23s} {len(group):9d} {per_f1[group].mean():8.4f}")


# ------------------------------------------------------------
# Every statute: F1, threshold, gold and predicted counts on dev
# ------------------------------------------------------------

tp = (Y_dev.astype(bool) & dev_pred).sum(axis=0)
gold = Y_dev.sum(axis=0).astype(int)
pred = dev_pred.sum(axis=0)

print(f"\n{'statute':>8s} {'train':>7s} {'dev gold':>9s} {'predicted':>10s} {'F1':>6s} {'threshold':>10s}")
for j in np.argsort(-POSITIVES):
    print(f"{LABELS[j]:8d} {int(POSITIVES[j]):7d} {gold[j]:9d} {int(pred[j]):10d} {per_f1[j]:6.3f} {THRESHOLDS[j]:10.2f}")

worst = np.argsort(per_f1)[:10]
print("\n10 worst statutes:", [(LABELS[j], round(float(per_f1[j]), 3)) for j in worst])

fn = (Y_dev.astype(bool) & ~dev_pred).sum(axis=0)
fp = (~Y_dev.astype(bool) & dev_pred).sum(axis=0)
print("most missed (false negatives):", [(LABELS[j], int(fn[j])) for j in np.argsort(-fn)[:10]])
print("most over-predicted (false positives):", [(LABELS[j], int(fp[j])) for j in np.argsort(-fp)[:10]])


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

results = {
    "run": RUN_NAME,
    "model": "Architecture-3 E-LegalFormer + AdaLoRA; mix of all layers; linear path (starts as logistic regression) + label attention over all tokens",
    "trained_on": "train only (dev used to pick the best epoch and the thresholds)",
    "input": {"chunk_len": CHUNK_LEN, "max_chunks": MAX_CHUNKS, "placeholders": PLACEHOLDERS, "pool_skip_spaces": POOL_SKIP_SPACES},
    "technique": {"method": "AdaLoRA", "init_rank": ADALORA_INIT_RANK, "target_rank": ADALORA_TARGET_RANK, "targets": ADALORA_TARGETS, "lr": LR_PEFT, "final_ranks": rank_summary(model)},
    "loss": {"bce_pos_weight_power": POS_WEIGHT_POWER, "pos_weight_max": POS_WEIGHT_MAX},
    "parameters_total": n_total, "parameters_trained_in_phase1_model": n_trained, "parameters_head": n_head,
    "best_epoch": best_epoch, "train_minutes": train_minutes, "peak_gpu_gb": peak_gpu_gb,
    "dev_predict_seconds": predict_seconds,
    "probe_by_layer": probe_results, "probe_best_layer": PROBE_BEST_LAYER,
    "start_logistic_regression": {k: v for k, v in START.items() if k != "per_statute_f1"},
    "layer_mix": {n: round(float(w), 4) for n, w in zip(LAYER_NAMES, model.layer_weights().detach())},
    "dev_threshold_0.5": {k: v for k, v in at_05.items() if k != "per_statute_f1"},
    "dev": {k: v for k, v in DEV_SCORES.items() if k != "per_statute_f1"},
    "at_least_one": AT_LEAST_ONE,
    "dev_by_frequency": by_frequency,
    "per_statute": {int(LABELS[j]): {"f1": float(per_f1[j]), "threshold": float(THRESHOLDS[j]),
                                     "train_docs": int(POSITIVES[j]), "dev_gold": int(gold[j]), "dev_predicted": int(pred[j])}
                    for j in range(NUM_LABELS)},
    "history": history,
}

with open(OUTPUT_DIR / "results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

np.save(OUTPUT_DIR / "dev_probabilities.npy", dev_probs.astype(np.float32))

print("\nSaved:", OUTPUT_DIR / "results.json", "and dev_probabilities.npy")


DEV (p5_arch3_adalora, best epoch 4, 10181 documents, predicted in 676s)
decision                                macro-F1  micro-F1  subset acc  macro-P  macro-R
threshold 0.5 for every statute           0.2770    0.3801      0.0184   0.2374   0.3543
threshold per statute                     0.2997    0.3819      0.0130   0.2697   0.3818

statutes per document: gold 3.92 | predicted 6.44

 train docs per statute  statutes  mean F1
                  0-499        30   0.2098
                500-999        27   0.2456
              1000-2999        27   0.3465
                >= 3000        16   0.4805

 statute   train  dev gold  predicted     F1  threshold
       5   11730      2936       3724  0.567       0.44
      34   10380      2596       2610  0.797       0.58
      97    7880      1970       3501  0.477       0.34
      43    7728      1932       3238  0.426       0.36
      39    7120      1779       2591  0.502       0.42
      12    4324      1086       2148  0.375       0.38


### CELL 15 · Predict test + write prediction.json + submission.zip

**In simple words:** Predict the statutes of every test document and write `submission.zip`.

**How it works:** `prediction.json` = `[{"id": "case_001", "pred": [1, 7, 23]}, …]` (unique, valid statute ids).

In [15]:
# ============================================================
# CELL 15 — PREDICT TEST + WRITE prediction.json + submission.zip
#   format: [{"id": "case_001", "pred": [1, 7, 23]}, ...]  (integers, unique, valid statute ids)
# ============================================================

t0 = time.time()
test_probs = predict_probs(model, test_docs)
test_pred = decide(test_probs, THRESHOLDS, AT_LEAST_ONE)
print(f"Predicted {len(test_docs)} test documents in {time.time() - t0:.0f}s")

submission = [{"id": d["id"], "pred": [int(LABELS[j]) for j in np.where(row)[0]]}
              for d, row in zip(test_docs, test_pred)]


# ------------------------------------------------------------
# Format checks
# ------------------------------------------------------------

valid = set(int(l) for l in LABELS)

assert len(submission) == len(test_docs)
assert [s["id"] for s in submission] == [d["id"] for d in test_docs]
assert all(isinstance(p, int) and p in valid for s in submission for p in s["pred"])
assert all(len(s["pred"]) == len(set(s["pred"])) for s in submission)

n_pred = np.array([len(s["pred"]) for s in submission])
print(f"✅ {len(submission)} predictions | statutes per document: mean {n_pred.mean():.2f}, "
      f"none {int((n_pred == 0).sum())}, max {n_pred.max()}")


PREDICTION_FILE = OUTPUT_DIR / "prediction.json"
ZIP_FILE = OUTPUT_DIR / "submission.zip"

with open(PREDICTION_FILE, "w", encoding="utf-8") as f:
    json.dump(submission, f)

np.save(OUTPUT_DIR / "test_probabilities.npy", test_probs.astype(np.float32))

with zipfile.ZipFile(ZIP_FILE, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(PREDICTION_FILE, arcname="prediction.json")        # file at the ROOT of the zip

with zipfile.ZipFile(ZIP_FILE) as zf:
    print("submission.zip:", zf.namelist())

print("Saved:", ZIP_FILE)


Predicted 13019 test documents in 853s
✅ 13019 predictions | statutes per document: mean 6.30, none 155, max 30
submission.zip: ['prediction.json']
Saved: /kaggle/working/p5_arch3_adalora/submission.zip
